In [1]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset, concatenate_datasets
import evaluate
import utils

<a name='1'></a>
## 1 - Import the Dataset

Bagian ini mempersiapkan dataset yang berasal dari dua sumber yaitu The First Certificate in English (FCE) corpus dan Birkbeck Misspelled Words

* Dataset FCE untuk Grammar Error Correction
* Dataset Birkbeck untuk Misspelled Correction

<a name='6-1'></a>
### 1.1 Grammar Error Dataset

load_dataset_from_file menghasilkan target_text dengan mengubah sebagian kata input_text berdasarkan edits(kata, index) untuk menghasilkan teks dengan grammar benar

In [2]:
train_dataset = utils.load_dataset_from_file(r"dataset/fce/json/fce.train.json")
eval_dataset = utils.load_dataset_from_file(r"dataset/fce/json/fce.dev.json")
test_dataset = utils.load_dataset_from_file(r"dataset/fce/json/fce.test.json")

df = pd.DataFrame(train_dataset,  columns=['input_text', 'target_text'])
print(df.sample(5))

                                             input_text  \
423   Dear Sir or Madam,\n\nI am writing to complain...   
685   Hello,\n\nI'm so glad I won this prize, but I ...   
388   16.12.00\n\nDear Nico,\n\nIts really nice to h...   
1733  Dear Sir or Madam,\n\nI am writing to giving  ...   
875   Dear Helen Ryan\n\nThank you for your letter i...   

                                            target_text  
423   Dear Sir or Madam,\n\nI am writing to complain...  
685   Hello,\n\nI'm so glad I won this prize, but I ...  
388   16.12.00\n\nDear Nico,\n\nIt's really nice to ...  
1733  Dear Sir or Madam,\n\nI am writing to give you...  
875   Dear Helen Ryan\n\nThank you for your letter i...  


<a name='6-1'></a>
### 1.2 Misspelled Dataset

make_sentence_pairs dan make_sentence_pairs1 (hanya berbeda kalimat template) menghasilkan kalimat dari kalimat template yang memiliki blank dan diisi oleh misspelled words (input_text) dan correct words (target_text)

In [3]:
file_path = r'dataset/missp.dat.txt'
word_pairs = utils.read_missp_file(file_path)
sentence_pairs = utils.make_sentence_pairs(word_pairs)
sentence_pairs1 = utils.make_sentence_pairs1(word_pairs)

df_train = pd.DataFrame(sentence_pairs, columns=['input_text', 'target_text'])
df_eval = pd.DataFrame(sentence_pairs1, columns=['input_text', 'target_text'])
print(df_train.sample(5))

misspelled_train_dataset = Dataset.from_pandas(df_train)
misspelled_eval_dataset = Dataset.from_pandas(df_eval)

split_ratio = 0.2
num_eval_samples = int(len(misspelled_eval_dataset) * split_ratio)
misspelled_eval_dataset_20 = misspelled_eval_dataset.select(range(num_eval_samples))

                                      input_text  \
12521         I often misspell the word ephisal.   
26136     In class, we practiced using recevber.   
2111   We used annylas during our group project.   
15178     Sometimes freb is spelled incorrectly.   
5972      During lunch, we talked about catalue.   

                                      target_text  
12521          I often misspell the word epistle.  
26136      In class, we practiced using received.  
2111   We used analysis during our group project.  
15178    Sometimes friend is spelled incorrectly.  
5972    During lunch, we talked about catalogues.  


<a name='1'></a>
## 2 - Load ProphetNet pre-trained Model from HuggingFace

In [2]:
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


<a name='1'></a>
## 3 - Preprocessing the Data

* Bagian ini memproses dataset dengan mengubah pasangan teks input-output (input_text, target_text) menjadi token ID yang bisa diproses oleh T5 untuk sequence-to-sequence learning
* max_target_length di set ke 128 untuk mengimbangi kemampuan GPU dan membuat model lebih terlatih untuk konteks input yang lebih pendek

In [5]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

combined_train = concatenate_datasets([train_dataset, misspelled_train_dataset])
combined_eval = concatenate_datasets([eval_dataset, misspelled_eval_dataset_20])
tokenized_train = combined_train.map(preprocess_function, batched=True)
tokenized_eval = combined_eval.map(preprocess_function, batched=True)

Map:   0%|          | 0/38249 [00:00<?, ? examples/s]/home/jupyter-c14220344@john.pet-5348a/nlp-main/prophet/venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:3959: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 7385/7385 [00:01<00:00, 6127.44 examples/s]


<a name='1'></a>
## 4 - Train the Model

In [6]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-grammar-typo-correction",
    learning_rate=4e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_dir='./logs',
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [9]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

/tmp/ipykernel_4157100/3182559150.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
500,0.303100
1000,0.475000
1500,0.557100
2000,0.525100
2500,0.521400
3000,0.508300
3500,0.498200
4000,0.473000
4500,0.460700
5000,0.454700


TrainOutput(global_step=47815, training_loss=0.31416702294441917, metrics={'train_runtime': 6395.2006, 'train_samples_per_second': 29.904, 'train_steps_per_second': 7.477, 'total_flos': 1.368970759300608e+16, 'train_loss': 0.31416702294441917, 'epoch': 5.0})

<a name='1'></a>
## 5 - Evaluate the Model using BLEU and ROUGE

In [3]:
model_dir = r"D:\Natural Language Processing\Project_Akhir\T5\checkpoint-47815"

tokenizer = T5Tokenizer.from_pretrained(model_dir)
model = T5ForConditionalGeneration.from_pretrained(model_dir)

In [4]:
def correct_grammar(sentence: str, max_len: int = 128) -> str:
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=max_len,
        num_beams=5,
        early_stopping=True
    )
    corrected_sentence = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected_sentence

In [5]:
with open("source.txt", "r", encoding="utf-8") as f:
    data = [line.strip() for line in f]

with open("pred.txt", "w", encoding="utf-8") as pred:
    
    for entry in data:
        pred.write(correct_grammar(entry).replace("\n", " ") + "\n")

In [7]:
with open("pred.txt", "r", encoding="utf-8") as f:
    predictions = [line.strip() for line in f]

with open("target.txt", "r", encoding="utf-8") as f:
    targets = [line.strip() for line in f]

assert len(predictions) == len(targets), "Jumlah prediksi dan referensi harus sama!"

references = [[ref] for ref in targets]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(predictions=predictions, references=references)
print(f"\nBLEU Score: {bleu_score['bleu']:.4f}")

rouge = evaluate.load("rouge")
rouge_score = rouge.compute(predictions=predictions, references=targets)
print("\nROUGE Scores:")
for key, value in rouge_score.items():
    print(f"{key}: {value:.4f}")


BLEU Score: 0.3723

ROUGE Scores:
rouge1: 0.7391
rouge2: 0.6694
rougeL: 0.7226
rougeLsum: 0.7220


<a name='1'></a>
## 6 - Try to Correct some Sentences!

In [9]:
file_path = r"D:\Natural Language Processing\Project_Akhir\dataset\input.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]

for idx, para in enumerate(paragraphs, 1):
    correction = correct_grammar(para)
    print(f"\nParagraf {idx}:")
    print("Input:")
    print(utils.wrap_text_by_words(para))
    print("\nKoreksi:")
    print(utils.wrap_text_by_words(correction))
    print("-" * 80)


Paragraf 1:
Input:
she have many friends and teacher

Koreksi:
She have many friends and teacher. She has many friends and teacher.
--------------------------------------------------------------------------------

Paragraf 2:
Input:
he is a senior docter

Koreksi:
he is a senior doctor. He is a senior doctor.
--------------------------------------------------------------------------------

Paragraf 3:
Input:
Many student thinks that to learn a forein language is difficult because they
haven't enough oportunity to practise. In the school, they usualy studies
grammar, but not speak much. Also, teacher doesn’t give advices how to improve
listening skills, which make more harder to understand native speakers. Some
have tryed to watch films without subtitel, but it not helped them much.

Koreksi:
Many students think that learning a foreign language is difficult because they
haven't enough opportunities to practise. In the school, they usually study
grammar, but not speak much. Also, the te